# 🧵 KaarigarAI — Artisan & Craft Marketplace Data Scraper
### Dataset Collection Pipeline for Craft Category Classifier & Multimodal XGBoost Price Predictor

This notebook runs in **Google Colab** to collect 5,000–10,000+ authentic Indian handloom, pottery, metalcraft, and handicraft listings with:
- Product Title & Cleaned Descriptions
- Standardized Craft Categories (`textile`, `pottery`, `woodcraft`, `metalcraft`, `jewellery`, `paintings`, `others`)
- Actual Market Price (₹ INR)
- GI-Tag Detection (Banarasi, Sambalpuri, Madhubani, Dhokra, etc.)
- Material Tiers & Technique Complexity Scores
- High-Res Image URLs + Optional Image Downloader

## 1. Setup & Install Dependencies

In [ ]:
# Install required packages
!pip install -q requests beautifulsoup4 pandas tqdm pillow matplotlib seaborn

## 2. (Optional) Mount Google Drive
Mount Google Drive if you want to save datasets and images directly to your Drive storage.

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/KaarigarAI_Data'
    os.makedirs(SAVE_DIR, exist_ok=True)
    print(f"Mounted Google Drive. Output folder: {SAVE_DIR}")
except Exception as e:
    SAVE_DIR = '/content/kaarigar_data'
    os.makedirs(SAVE_DIR, exist_ok=True)
    print(f"Local directory fallback: {SAVE_DIR}")

## 3. Core Scraper Engine & Feature Engineering Rules

In [ ]:
import re
import time
import random
import requests
from bs4 import BeautifulSoup
import pandas as pd
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# GI-tagged craft identifiers
GI_TAG_CRAFTS = [
    'banarasi', 'kanjeevaram', 'kanchipuram', 'madhubani', 'warli', 'phulkari',
    'kantha', 'dhokra', 'bidri', 'chanderi', 'sambalpuri', 'kota doria', 'pochampally',
    'paithani', 'bandhani', 'patola', 'pattachitra', 'kullu shawl', 'muga silk',
    'mysore silk', 'tussar silk', 'jamdani', 'bagh print', 'sanganeri', 'kalamkari',
    'blue pottery', 'tanjore painting', 'bastar iron craft', 'terracotta bankura',
    'kondapalli', 'chanapatna', 'moradabad brass', 'saharanpur wood'
]

CATEGORY_TAXONOMY = {
    'textile': ['saree', 'sari', 'dupatta', 'shawl', 'stole', 'fabric', 'handloom', 'ikat', 'silk', 'cotton', 'linen', 'khadi', 'kurta', 'kantha', 'phulkari', 'chanderi', 'banarasi', 'bandhani'],
    'pottery': ['pottery', 'terracotta', 'clay', 'ceramic', 'vase', 'kulhad', 'matka', 'planter', 'earthenware', 'blue pottery'],
    'woodcraft': ['wood', 'wooden', 'teak', 'sheesham', 'rosewood', 'carving', 'channapatna', 'toy', 'sculpture', 'bamboo', 'cane'],
    'metalcraft': ['brass', 'bronze', 'copper', 'dhokra', 'dokra', 'bidri', 'bell metal', 'iron craft', 'moradabad', 'diya', 'statue'],
    'jewellery': ['jewellery', 'jewelry', 'necklace', 'earring', 'bangle', 'pendant', 'tribal jewelry', 'silver', 'lac', 'filigree', 'meenakari'],
    'paintings': ['painting', 'madhubani', 'warli', 'pattachitra', 'tanjore', 'gond', 'canvas', 'miniature painting'],
}

MATERIAL_KEYWORDS = {
    'premium': ['silk', 'pure silk', 'mulberry silk', 'tussar', 'muga', 'teak', 'rosewood', 'silver', 'brass', 'bronze', 'bidri'],
    'mid': ['wool', 'linen', 'copper', 'bell metal', 'sheesham', 'lac', 'dhokra', 'chanderi', 'bamboo'],
    'basic': ['cotton', 'jute', 'clay', 'terracotta', 'earthen', 'softwood']
}

TECHNIQUES = ['handwoven', 'handstitched', 'handpainted', 'handcarved', 'carved', 'embroidered', 'block-printed', 'natural-dyed', 'hand-spun', 'tie and dye', 'filigree']

USER_AGENTS = [
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36'
]

def clean_html_text(text):
    if not text: return ''
    clean = re.sub(r'<[^>]+>', ' ', str(text))
    return re.sub(r'\s+', ' ', clean).strip()

def infer_category(title, desc, raw_cat=''):
    comb = f"{title} {desc} {raw_cat}".lower()
    for cat, kws in CATEGORY_TAXONOMY.items():
        if any(k in comb for k in kws):
            return cat
    return 'others'

def has_gi_tag(title, desc):
    comb = f"{title} {desc}".lower()
    return int(any(craft in comb for craft in GI_TAG_CRAFTS))

def get_material_tier(title, desc):
    comb = f"{title} {desc}".lower()
    if any(m in comb for m in MATERIAL_KEYWORDS['premium']): return 2
    if any(m in comb for m in MATERIAL_KEYWORDS['mid']): return 1
    return 0

def get_technique_score(title, desc):
    comb = f"{title} {desc}".lower()
    return min(sum(1 for t in TECHNIQUES if t in comb), 3)

def parse_price(price_val):
    if price_val is None: return None
    cleaned = re.sub(r'[^\d.]', '', str(price_val))
    try:
        v = float(cleaned)
        return v if v > 0 else None
    except:
        return None

## 4. Run Multi-Store Scraping Engine

In [ ]:
TARGET_DOMAINS = [
    'https://gocoop.com',
    'https://www.itokri.com',
    'https://www.theloom.in',
    'https://gaatha.com'
]

def scrape_all_stores(domains, max_pages_per_store=12):
    records = []
    session = requests.Session()
    
    for domain in domains:
        print(f"\n🔍 Scraping store: {domain} ...")
        for page in range(1, max_pages_per_store + 1):
            url = f"{domain.rstrip('/')}/products.json"
            headers = {'User-Agent': random.choice(USER_AGENTS)}
            
            try:
                resp = session.get(url, params={'limit': 250, 'page': page}, headers=headers, timeout=15)
                time.sleep(random.uniform(0.6, 1.2))
                if resp.status_code != 200:
                    print(f"  ⚠️ Finished or status {resp.status_code} at page {page}")
                    break
                
                products = resp.json().get('products', [])
                if not products:
                    print(f"  ✅ Reached end of catalog at page {page}")
                    break
                
                for p in products:
                    title = p.get('title', '').strip()
                    desc = clean_html_text(p.get('body_html', ''))
                    raw_cat = p.get('product_type', '')
                    tags = p.get('tags', [])
                    tags_str = ', '.join(tags) if isinstance(tags, list) else str(tags)
                    
                    variants = p.get('variants', [])
                    price = parse_price(variants[0].get('price')) if variants else None
                    compare_at = parse_price(variants[0].get('compare_at_price')) if variants else None
                    
                    images = p.get('images', [])
                    img_url = images[0].get('src') if images else None
                    
                    if not title or not price or not img_url:
                        continue
                    
                    cat = infer_category(title, desc, raw_cat)
                    gi = has_gi_tag(title, desc) or has_gi_tag(tags_str, '')
                    mat_tier = get_material_tier(title, desc)
                    tech = get_technique_score(title, desc)
                    
                    records.append({
                        'source': domain,
                        'source_id': str(p.get('id', '')),
                        'title': title,
                        'category': cat,
                        'raw_category': raw_cat,
                        'price': price,
                        'compare_at_price': compare_at,
                        'description': desc[:1000],
                        'tags': tags_str,
                        'image_url': img_url,
                        'has_gi_tag': gi,
                        'material_tier': mat_tier,
                        'technique_score': tech,
                        'description_length': len(desc),
                    })
                print(f"  → Page {page}: extracted {len(products)} products (Total: {len(records)})")
            except Exception as e:
                print(f"  ❌ Error on {url}: {e}")
                break
                
    return pd.DataFrame(records)

# Execute scraping
df_raw = scrape_all_stores(TARGET_DOMAINS, max_pages_per_store=12)
print(f"\n🎉 Total Raw Listings Collected: {len(df_raw)}")

## 5. Clean, Validate, and Filter Dataset

In [ ]:
def clean_dataset(df, min_price=50.0, max_price=50000.0):
    init_len = len(df)
    df_clean = df.dropna(subset=['title', 'price', 'image_url']).copy()
    df_clean['price'] = pd.to_numeric(df_clean['price'], errors='coerce')
    df_clean = df_clean[(df_clean['price'] >= min_price) & (df_clean['price'] <= max_price)]
    df_clean = df_clean.drop_duplicates(subset=['title', 'price'])
    df_clean = df_clean.drop_duplicates(subset=['image_url'])
    
    print(f"Cleaning Summary: {len(df_clean)} / {init_len} listings retained ({len(df_clean)/max(init_len,1)*100:.1f}%)")
    return df_clean

df_cleaned = clean_dataset(df_raw)

# Save to CSV
raw_path = os.path.join(SAVE_DIR, 'craft_listings_raw.csv')
clean_path = os.path.join(SAVE_DIR, 'craft_listings_cleaned.csv')

df_raw.to_csv(raw_path, index=False)
df_cleaned.to_csv(clean_path, index=False)
print(f"💾 Raw data saved to: {raw_path}")
print(f"💾 Cleaned ML data saved to: {clean_path}")

## 6. Dataset Distribution Analysis & Verification

In [ ]:
plt.figure(figsize=(14, 5))

# 1. Category Distribution
plt.subplot(1, 2, 1)
cat_counts = df_cleaned['category'].value_counts()
sns.barplot(x=cat_counts.values, y=cat_counts.index, palette='viridis')
plt.title('Craft Category Distribution')
plt.xlabel('Number of Listings')

# 2. Price Distribution (Log Scale)
plt.subplot(1, 2, 2)
sns.histplot(df_cleaned['price'], bins=40, kde=True, color='#E8873A')
plt.title('Price Distribution (INR ₹)')
plt.xlabel('Price (₹)')
plt.tight_layout()
plt.show()

print("\nCategory Breakdown:")
print(df_cleaned['category'].value_counts(normalize=True).mul(100).round(2).to_string())

## 7. (Optional) Batch Image Downloader for Model Training
Downloads product photos categorized into folders (`dataset_images/<category>/img_xxxxxx.jpg`).

In [ ]:
DOWNLOAD_IMAGES = True # Set to True to download images
MAX_IMAGES_TO_DOWNLOAD = 3000
IMG_DIR = os.path.join(SAVE_DIR, 'dataset_images')

if DOWNLOAD_IMAGES:
    os.makedirs(IMG_DIR, exist_ok=True)
    session = requests.Session()
    session.headers.update({'User-Agent': random.choice(USER_AGENTS)})
    
    subset = df_cleaned.head(MAX_IMAGES_TO_DOWNLOAD).copy()
    local_paths = []
    
    print(f"Downloading {len(subset)} images into {IMG_DIR}...")
    for idx, row in tqdm(subset.iterrows(), total=len(subset)):
        cat_folder = os.path.join(IMG_DIR, row['category'])
        os.makedirs(cat_folder, exist_ok=True)
        img_file = os.path.join(cat_folder, f"img_{idx:06d}.jpg")
        
        if os.path.exists(img_file):
            local_paths.append(img_file)
            continue
        
        try:
            r = session.get(row['image_url'], timeout=10)
            if r.status_code == 200 and len(r.content) > 1024:
                with open(img_file, 'wb') as f:
                    f.write(r.content)
                local_paths.append(img_file)
            else:
                local_paths.append(None)
        except:
            local_paths.append(None)
            
    subset['local_image_path'] = local_paths
    img_dataset_path = os.path.join(SAVE_DIR, 'craft_listings_with_images.csv')
    subset.dropna(subset=['local_image_path']).to_csv(img_dataset_path, index=False)
    print(f"\n✅ Image download complete! Saved catalog with local paths to {img_dataset_path}")